# Taller 2 - Punto 3: Fine-tuning de xlm-roberta-large para Analisis de Sentimientos

Este notebook implementa el fine-tuning del modelo xlm-roberta-large de Facebook para clasificacion de sentimientos en tweets en espanol.

## Objetivos:
1. Autenticacion en Hugging Face
2. Carga y preprocesamiento del dataset de Tweets
3. Tokenizacion con xlm-roberta-large
4. Configuracion y entrenamiento del modelo con class weights
5. Evaluacion con metricas de clasificacion (Accuracy, F1, Precision, Recall)
6. Visualizacion del progreso de entrenamiento
7. Subida del modelo a Hugging Face Hub

Este punto requiere GPU para entrenar xlm-roberta-large de forma eficiente.

## 0. Instalacion de Dependencias

**NOTA:** Este notebook está diseñado para ejecutarse con la extensión de VS Code para Google Colab.

- Los archivos (`helpers.py`, datasets) están en tu máquina local
- El código se ejecuta en el kernel de Google Colab
- Las instalaciones se hacen en el entorno remoto de Colab

In [2]:
# Instalacion de librerias para Google Colab
# Este notebook usa la extension de VS Code para Colab
# Los archivos estan en local, pero el codigo se ejecuta en Colab

import sys

print("="*80)
print("CONFIGURACION: VS Code + Google Colab Extension")
print("="*80)
print("\nEste notebook se ejecuta en el kernel de Google Colab")
print("Los archivos (helpers.py, datasets) estan en tu maquina local")
print("\nInstalando dependencias en el entorno de Colab...")
print("Esto puede tomar varios minutos...\n")

# Instalar librerias principales en Colab
!pip install -q transformers==4.57.1
!pip install -q datasets==4.4.1
!pip install -q torch torchvision torchaudio
!pip install -q huggingface-hub
!pip install -q scikit-learn
!pip install -q pandas numpy
!pip install -q matplotlib seaborn
!pip install -q accelerate

print("\n" + "="*80)
print("✓ Instalacion completada en Colab")
print("="*80)

# Verificar GPU (en Colab)
import torch
if torch.cuda.is_available():
    print(f"\n✓ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠ WARNING: GPU no disponible!")
    print("⚠ Este modelo requiere GPU para entrenar eficientemente")
    print("\n🔧 Para activar GPU en Colab:")
    print("   1. En la web de Colab: Runtime > Change runtime type > GPU")
    print("   2. En VS Code: Comando 'Colab: Change Runtime Type'")

# Verificar archivos locales
import os
print("\n" + "="*80)
print("VERIFICACION DE ARCHIVOS LOCALES")
print("="*80)

# Verificar helpers.py
if os.path.exists('helpers.py'):
    print("\n✓ helpers.py encontrado")
else:
    print("\n⚠ helpers.py no encontrado")
    print("  Asegurate de que el archivo existe en tu workspace local")

# Verificar datasets
tass_files = ["./input/TASS/cr-tass.csv", "./input/TASS/cr1-tass.csv"]
tass_ok = all(os.path.exists(f) for f in tass_files)
if tass_ok:
    print("✓ Datasets TASS encontrados")
    for f in tass_files:
        size = os.path.getsize(f) / 1024
        print(f"  - {os.path.basename(f)} ({size:.1f} KB)")
else:
    print("⚠ Datasets TASS no encontrados")
    print("  Asegurate de tener los archivos en ./input/TASS/")

print("\n" + "="*80)
print("ENTORNO CONFIGURADO")
print("="*80)
print("\n✓ Librerias instaladas en Colab (remoto)")
print("✓ Archivos disponibles localmente")
print("✓ Listo para entrenar")
print("\n" + "="*80)

CONFIGURACION: VS Code + Google Colab Extension

Este notebook se ejecuta en el kernel de Google Colab
Los archivos (helpers.py, datasets) estan en tu maquina local

Instalando dependencias en el entorno de Colab...
Esto puede tomar varios minutos...


✓ Instalacion completada en Colab

✓ Instalacion completada en Colab

✓ GPU disponible: Tesla T4
✓ Memoria GPU: 15.83 GB

VERIFICACION DE ARCHIVOS LOCALES

⚠ helpers.py no encontrado
  Asegurate de que el archivo existe en tu workspace local
⚠ Datasets TASS no encontrados
  Asegurate de tener los archivos en ./input/TASS/

ENTORNO CONFIGURADO

✓ Librerias instaladas en Colab (remoto)
✓ Archivos disponibles localmente
✓ Listo para entrenar


✓ GPU disponible: Tesla T4
✓ Memoria GPU: 15.83 GB

VERIFICACION DE ARCHIVOS LOCALES

⚠ helpers.py no encontrado
  Asegurate de que el archivo existe en tu workspace local
⚠ Datasets TASS no encontrados
  Asegurate de tener los archivos en ./input/TASS/

ENTORNO CONFIGURADO

✓ Librerias instaladas en 

## 3.1 Configuracion Inicial y Autenticacion en Hugging Face

**IMPORTANTE:** Hay dos formas de autenticarte:

1. **Recomendado:** Ejecutar `huggingface-cli login` en tu terminal local antes de iniciar el kernel
2. **Alternativa:** Reemplazar `TU_TOKEN_AQUI` con tu token directamente en el código

Puedes obtener tu token en: https://huggingface.co/settings/tokens

In [3]:
from huggingface_hub import login
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("AUTENTICACION EN HUGGING FACE")
print("="*80)

# OPCION 1: Intentar usar credenciales ya guardadas (si ejecutaste huggingface-cli login)
print("\n[Opción 1] Intentando usar credenciales guardadas...")
try:
    login(token=None)  # Intenta usar el token guardado
    print("✓ Autenticación exitosa con credenciales guardadas")
    print("\n" + "="*80)
except Exception as e:
    print(f"✗ No se encontraron credenciales guardadas")
    print(f"  Detalle: {e}")
    
    # OPCION 2: Usar token directo
    print("\n[Opción 2] Usando token directo...")
    
    # REEMPLAZA ESTO CON TU TOKEN DE HUGGINGFACE
    HF_TOKEN = "TU_TOKEN_AQUI"
    
    if HF_TOKEN == "TU_TOKEN_AQUI":
        print("\n" + "="*80)
        print("⚠ ERROR: TOKEN NO CONFIGURADO")
        print("="*80)
        print("\nPara autenticarte, elige una opción:")
        print("\n1. RECOMENDADO - Ejecutar en tu terminal local:")
        print("   huggingface-cli login")
        print("   (Esto guarda el token de forma segura)")
        print("\n2. ALTERNATIVA - Reemplazar el token en esta celda:")
        print("   HF_TOKEN = 'tu_token_aqui'")
        print("\n3. Obtener token en:")
        print("   https://huggingface.co/settings/tokens")
        print("   (Necesitas permisos de escritura para subir modelos)")
        print("\n" + "="*80)
    else:
        try:
            login(HF_TOKEN)
            print("✓ Autenticación exitosa con token directo")
            print("\n" + "="*80)
        except Exception as e2:
            print(f"✗ Error en autenticación: {e2}")
            print("\nVerifica que:")
            print("  1. El token sea válido")
            print("  2. Tenga permisos de escritura (si vas a subir modelos)")
            print("  3. Esté copiado correctamente (sin espacios extras)")
            print("\n" + "="*80)

Advertencia: Archivo .env no encontrado. Usando token por defecto.
Error en autenticacion: Invalid user token.
Por favor, verifica tu token de Hugging Face
Error en autenticacion: Invalid user token.
Por favor, verifica tu token de Hugging Face


In [ ]:
# Imports necesarios
import os
import re
import json
import pandas as pd
import numpy as np
import torch
from torch import nn
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback
)
import matplotlib.pyplot as plt
import seaborn as sns

# Importar funciones del archivo helpers
from helpers import (
    ensure_directories,
    save_experiment_results,
    load_experiment_results,
    save_training_history,
    load_training_history,
    plot_training_history
)

print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3.2 Configuracion de Directorios y Parametros

In [ ]:
# Configuracion de directorios
DATA_DIR = "./input/TASS/"
OUTPUT_DIR = "./output/punto3/"
MODELS_DIR = "./models/punto3/"
RESULTS_DIR = "./results/"

# Crear directorios necesarios
ensure_directories(3)

# Configuracion del modelo base
MODEL_NAME = "xlm-roberta-large"
MODEL_CHECKPOINT = "FacebookAI/xlm-roberta-large"
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10  # Segun el taller: 10 epocas
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500

# Configuracion de evaluacion
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"
METRIC_FOR_BEST_MODEL = "f1"
EARLY_STOPPING_PATIENCE = 3

# Batch sizes a experimentar segun el taller: 4, 8 y 16
BATCH_SIZES = [4, 8, 16]

print("="*80)
print("CONFIGURACION DEL PUNTO 3 - FINE-TUNING XLM-ROBERTA-LARGE")
print("="*80)
print(f"\nModelo base: {MODEL_NAME}")
print(f"Checkpoint: {MODEL_CHECKPOINT}")
print(f"Max Length: {MAX_LENGTH}")
print(f"\nHIPERPARAMETROS:")
print(f"  - Batch Sizes a experimentar: {BATCH_SIZES}")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Numero de Epocas: {NUM_EPOCHS}")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - Warmup Steps: {WARMUP_STEPS}")
print(f"  - Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"  - Metric for Best Model: {METRIC_FOR_BEST_MODEL}")
print(f"\nDIRECTORIOS:")
print(f"  - Data: {DATA_DIR}")
print(f"  - Models: {MODELS_DIR}")
print(f"  - Output: {OUTPUT_DIR}")
print(f"  - Results: {RESULTS_DIR}")
print(f"\n{'='*80}")
print(f"Se realizaran {len(BATCH_SIZES)} experimentos independientes")
print(f"{'='*80}")

## 3.3 Carga y Preprocesamiento de Datos

Cargamos los datasets de tweets con sentimientos (N=Negativo, P=Positivo).

In [ ]:
def clean_text(text):
    """
    Limpia y normaliza el texto de tweets
    """
    if not isinstance(text, str):
        return ""
    
    # Convertir a minusculas
    text = text.lower()
    
    # Eliminar URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Eliminar menciones de usuario
    text = re.sub(r'@\w+', '', text)
    
    # Eliminar hashtags (mantener el texto)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # Eliminar caracteres especiales y numeros
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    
    # Eliminar espacios multiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Cargar datasets
print("Cargando datasets...")
df1 = pd.read_csv(f"{DATA_DIR}cr-tass.csv")
df2 = pd.read_csv(f"{DATA_DIR}cr1-tass.csv")

print(f"\nDataset 1 (cr-tass.csv): {len(df1)} muestras")
print(f"Dataset 2 (cr1-tass.csv): {len(df2)} muestras")

# Combinar datasets
df = pd.concat([df1, df2], ignore_index=True)
print(f"\nDataset combinado: {len(df)} muestras")

# Mostrar distribucion de clases original
print("\nDistribucion de clases (original):")
print(df['label'].value_counts())

# Filtrar solo clases N y P
df = df[df['label'].isin(['N', 'P'])].copy()
print(f"\nDespues de filtrar N y P: {len(df)} muestras")

# Renombrar columnas para facilitar el uso
df = df.rename(columns={'sentencia original': 'text', 'label': 'sentiment'})

# Limpiar textos
print("\nLimpiando textos...")
df['text_clean'] = df['text'].apply(clean_text)

# Eliminar textos vacios
df = df[df['text_clean'].str.len() > 0].copy()
print(f"Despues de eliminar textos vacios: {len(df)} muestras")

# Convertir etiquetas a numeros: N=0, P=1
label_map = {'N': 0, 'P': 1}
df['label'] = df['sentiment'].map(label_map)

# Mostrar distribucion final
print("\nDistribucion de clases (final):")
print(df['label'].value_counts())
print(f"\nClase 0 (Negativo): {(df['label']==0).sum()} muestras ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Clase 1 (Positivo): {(df['label']==1).sum()} muestras ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Mostrar ejemplos
print("\nEjemplos de tweets procesados:")
for i in range(3):
    print(f"\n{i+1}. Sentimiento: {df.iloc[i]['sentiment']}")
    print(f"   Original: {df.iloc[i]['text'][:100]}")
    print(f"   Limpio: {df.iloc[i]['text_clean'][:100]}")

## 3.4 Division de Datos y Tokenizacion

In [ ]:
# Dividir en train y test (80-20)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text_clean'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Dividir train en train y validation (80-20 del train, es decir 64-16-20 total)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

print("Division de datos:")
print(f"- Train: {len(train_texts)} muestras")
print(f"- Validation: {len(val_texts)} muestras")
print(f"- Test: {len(test_texts)} muestras")

# Verificar distribucion de clases
print("\nDistribucion de clases:")
print(f"Train - N: {train_labels.count(0)}, P: {train_labels.count(1)}")
print(f"Val   - N: {val_labels.count(0)}, P: {val_labels.count(1)}")
print(f"Test  - N: {test_labels.count(0)}, P: {test_labels.count(1)}")

# Cargar tokenizer
print(f"\nCargando tokenizer: {MODEL_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Tokenizer cargado exitosamente")

# Funcion de tokenizacion
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

# Crear datasets de HuggingFace
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})
test_dataset = Dataset.from_dict({'text': test_texts, 'label': test_labels})

# Tokenizar datasets
print("\nTokenizando datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Establecer formato
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenizacion completada")

# Mostrar ejemplo tokenizado
print("\nEjemplo de tokenizacion:")
print(f"Texto: {train_texts[0][:100]}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(train_dataset[0]['input_ids'])[:20]}")

## 3.5 Configuracion del Modelo con Class Weights

Configuramos el modelo con pesos de clase para manejar el desbalanceo de datos.

In [ ]:
# Calcular class weights para balancear clases
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"Class weights calculados:")
print(f"  Clase 0 (Negativo): {class_weights[0]:.4f}")
print(f"  Clase 1 (Positivo): {class_weights[1]:.4f}")
print("\nEstos pesos se usaran en el WeightedLossTrainer para balancear las clases")

## 3.6 Funciones de Metricas y Callbacks

In [ ]:
# Funcion para calcular metricas
def compute_metrics(eval_pred):
    """
    Calcula accuracy, F1, precision y recall
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    precision = precision_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Callback para guardar historial de entrenamiento
class MetricsHistoryCallback(TrainerCallback):
    def __init__(self):
        self.history = {
            'train_loss': [],
            'eval_loss': [],
            'eval_accuracy': [],
            'eval_f1': [],
            'eval_precision': [],
            'eval_recall': [],
            'epoch': []
        }
    
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        """Guardar metricas despues de cada evaluacion"""
        self.history['epoch'].append(state.epoch)
        self.history['eval_loss'].append(metrics.get('eval_loss', None))
        self.history['eval_accuracy'].append(metrics.get('eval_accuracy', None))
        self.history['eval_f1'].append(metrics.get('eval_f1', None))
        self.history['eval_precision'].append(metrics.get('eval_precision', None))
        self.history['eval_recall'].append(metrics.get('eval_recall', None))
    
    def on_log(self, args, state, control, logs, **kwargs):
        """Guardar loss de entrenamiento"""
        if 'loss' in logs:
            self.history['train_loss'].append(logs['loss'])

# Definir Trainer personalizado con weighted loss
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss con class weights
        Nota: num_items_in_batch es un parametro nuevo en transformers 4.x
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Aplicar weighted cross entropy loss
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
        
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("Funciones de metricas y callbacks configurados:")
print("  - compute_metrics: Calcula accuracy, F1, precision y recall")
print("  - MetricsHistoryCallback: Guarda el historial de entrenamiento")
print("  - WeightedLossTrainer: Trainer con loss ponderado por clase")
print("\nNota: WeightedLossTrainer compatible con transformers 4.x (soporta num_items_in_batch)")

## 3.6 Funciones de Metricas y Callbacks

Configuramos las funciones para calcular metricas y callbacks para monitorear el entrenamiento.

In [ ]:
# Diccionario para guardar todos los resultados
all_experiments = {}

# Iterar sobre cada batch size
for batch_size in BATCH_SIZES:
    print("\n" + "="*100)
    print(f"EXPERIMENTO: BATCH_SIZE = {batch_size}")
    print("="*100)
    
    # Crear callback para este experimento
    metrics_callback = MetricsHistoryCallback()
    
    # Cargar modelo fresco para cada experimento
    print(f"\nCargando modelo: {MODEL_CHECKPOINT}")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=2,
        problem_type="single_label_classification"
    )
    print(f"Modelo cargado - Parametros: {sum(p.numel() for p in model.parameters()):,}")
    
    # Configurar argumentos de entrenamiento
    experiment_output_dir = f"{OUTPUT_DIR}batch_size_{batch_size}/"
    os.makedirs(experiment_output_dir, exist_ok=True)
    
    training_args = TrainingArguments(
        output_dir=experiment_output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        logging_dir=f"{experiment_output_dir}logs/",
        logging_steps=50,
        eval_strategy=EVAL_STRATEGY,  # Cambiado de evaluation_strategy
        save_strategy=SAVE_STRATEGY,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST_MODEL,
        greater_is_better=True,
        save_total_limit=3,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=42
    )
    
    print(f"\nTrainingArguments configurados:")
    print(f"  - Batch size: {batch_size}")
    print(f"  - Epochs: {NUM_EPOCHS}")
    print(f"  - Learning rate: {LEARNING_RATE}")
    print(f"  - Output dir: {experiment_output_dir}")
    print(f"  - FP16: {torch.cuda.is_available()}")
    
    # Crear trainer con weighted loss
    trainer = WeightedLossTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
            metrics_callback
        ]
    )
    
    print(f"\nTrainer configurado con weighted loss y early stopping")
    
    # Entrenar modelo
    print(f"\n{'='*80}")
    print(f"INICIANDO ENTRENAMIENTO - Batch Size: {batch_size}")
    print(f"{'='*80}")
    
    train_result = trainer.train()
    
    print(f"\n{'='*80}")
    print(f"ENTRENAMIENTO COMPLETADO - Batch Size: {batch_size}")
    print(f"{'='*80}")
    print(f"  - Training loss: {train_result.training_loss:.4f}")
    print(f"  - Training runtime: {train_result.metrics['train_runtime']:.2f} segundos")
    print(f"  - Samples/second: {train_result.metrics['train_samples_per_second']:.2f}")
    
    # Evaluar en test set
    print(f"\nEvaluando en test set...")
    test_results = trainer.evaluate(test_dataset)
    
    print(f"\nResultados en Test Set:")
    print(f"  - Loss: {test_results['eval_loss']:.4f}")
    print(f"  - Accuracy: {test_results['eval_accuracy']:.4f}")
    print(f"  - F1 Score: {test_results['eval_f1']:.4f}")
    print(f"  - Precision: {test_results['eval_precision']:.4f}")
    print(f"  - Recall: {test_results['eval_recall']:.4f}")
    
    # Obtener predicciones para matriz de confusion
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    cm = confusion_matrix(test_labels, pred_labels)
    
    # Calcular tasas adicionales
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Guardar modelo
    model_save_path = f"{MODELS_DIR}batch_size_{batch_size}/"
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"\nModelo guardado en: {model_save_path}")
    
    # Guardar historial y resultados
    experiment_results = {
        'batch_size': batch_size,
        'history': metrics_callback.history,
        'train_result': {
            'training_loss': float(train_result.training_loss),
            'train_runtime': float(train_result.metrics['train_runtime']),
            'train_samples_per_second': float(train_result.metrics['train_samples_per_second'])
        },
        'test_results': {
            'loss': float(test_results['eval_loss']),
            'accuracy': float(test_results['eval_accuracy']),
            'f1': float(test_results['eval_f1']),
            'precision': float(test_results['eval_precision']),
            'recall': float(test_results['eval_recall']),
            'specificity': float(specificity),
            'sensitivity': float(sensitivity)
        },
        'confusion_matrix': cm.tolist(),
        'model_path': model_save_path
    }
    
    all_experiments[f'batch_size_{batch_size}'] = experiment_results
    
    # Guardar resultados individuales
    save_experiment_results(
        experiment_name=f"punto3_batch_size_{batch_size}",
        results=experiment_results,
        punto=3
    )
    
    # Limpiar memoria GPU
    del model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print(f"\n{'='*100}")
    print(f"EXPERIMENTO COMPLETADO - Batch Size: {batch_size}")
    print(f"{'='*100}\n")

print("\n" + "="*100)
print("TODOS LOS EXPERIMENTOS COMPLETADOS")
print("="*100)

## 3.7 Experimentos con Diferentes Batch Sizes

Modelo con 10 epocas y 3 batch sizes: 4, 8 y 16.

In [ ]:
# Crear DataFrame con resultados comparativos
comparison_data = []
for exp_name, exp_results in all_experiments.items():
    batch_size = exp_results['batch_size']
    test_res = exp_results['test_results']
    train_res = exp_results['train_result']
    
    comparison_data.append({
        'Batch Size': batch_size,
        'Test Accuracy': test_res['accuracy'],
        'Test F1': test_res['f1'],
        'Test Precision': test_res['precision'],
        'Test Recall': test_res['recall'],
        'Test Loss': test_res['loss'],
        'Specificity': test_res['specificity'],
        'Sensitivity': test_res['sensitivity'],
        'Training Loss': train_res['training_loss'],
        'Training Time (s)': train_res['train_runtime'],
        'Samples/sec': train_res['train_samples_per_second']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Batch Size')

print("="*100)
print("COMPARACION DE RESULTADOS - TODOS LOS EXPERIMENTOS")
print("="*100)
print("\n", comparison_df.to_string(index=False))

# Identificar mejor modelo por F1
best_idx = comparison_df['Test F1'].idxmax()
best_batch = comparison_df.loc[best_idx, 'Batch Size']

print(f"\n{'='*100}")
print(f"MEJOR MODELO: Batch Size = {best_batch}")
print(f"{'='*100}")
print(f"  - F1 Score: {comparison_df.loc[best_idx, 'Test F1']:.4f}")
print(f"  - Accuracy: {comparison_df.loc[best_idx, 'Test Accuracy']:.4f}")
print(f"  - Precision: {comparison_df.loc[best_idx, 'Test Precision']:.4f}")
print(f"  - Recall: {comparison_df.loc[best_idx, 'Test Recall']:.4f}")

# Visualizaciones comparativas
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Comparacion de metricas principales
ax = axes[0, 0]
metrics = ['Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']
x = np.arange(len(comparison_df))
width = 0.2
for i, metric in enumerate(metrics):
    ax.bar(x + i*width, comparison_df[metric], width, label=metric.replace('Test ', ''))
ax.set_xlabel('Batch Size')
ax.set_ylabel('Score')
ax.set_title('Comparacion de Metricas por Batch Size')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df['Batch Size'])
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Training Loss comparison
ax = axes[0, 1]
ax.bar(comparison_df['Batch Size'], comparison_df['Training Loss'], color='coral')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss por Batch Size')
ax.grid(True, alpha=0.3)

# 3. Training Time comparison
ax = axes[1, 0]
ax.bar(comparison_df['Batch Size'], comparison_df['Training Time (s)'], color='skyblue')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Tiempo (segundos)')
ax.set_title('Tiempo de Entrenamiento por Batch Size')
ax.grid(True, alpha=0.3)

# 4. Specificity vs Sensitivity
ax = axes[1, 1]
ax.plot(comparison_df['Batch Size'], comparison_df['Specificity'], 'o-', label='Specificity', linewidth=2)
ax.plot(comparison_df['Batch Size'], comparison_df['Sensitivity'], 's-', label='Sensitivity', linewidth=2)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Score')
ax.set_title('Specificity vs Sensitivity por Batch Size')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
comparison_plot_path = f"{OUTPUT_DIR}experiments_comparison.png"
plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
print(f"\nGrafica comparativa guardada en: {comparison_plot_path}")
plt.show()

# Guardar comparacion en CSV
comparison_csv_path = f"{RESULTS_DIR}punto3_experiments_comparison.csv"
comparison_df.to_csv(comparison_csv_path, index=False)
print(f"Tabla comparativa guardada en: {comparison_csv_path}")

## 3.8 Visualizacion del Historial de Entrenamiento

Visualizamos el progreso de entrenamiento para cada experimento y la matriz de confusion del mejor modelo.

In [ ]:
# Visualizar historial de entrenamiento para cada experimento
for exp_name, exp_results in all_experiments.items():
    batch_size = exp_results['batch_size']
    history = exp_results['history']
    
    print(f"\n{'='*80}")
    print(f"HISTORIAL DE ENTRENAMIENTO - Batch Size: {batch_size}")
    print(f"{'='*80}")
    
    # Crear figura con subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Historial de Entrenamiento - Batch Size {batch_size}', fontsize=16)
    
    # Loss
    if history['eval_loss']:
        ax = axes[0, 0]
        epochs = history['epoch']
        ax.plot(epochs, history['eval_loss'], 'o-', label='Validation Loss', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Loss durante el entrenamiento')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Accuracy
    if history['eval_accuracy']:
        ax = axes[0, 1]
        ax.plot(epochs, history['eval_accuracy'], 'o-', label='Validation Accuracy', color='green', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_title('Accuracy durante el entrenamiento')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # F1 Score
    if history['eval_f1']:
        ax = axes[1, 0]
        ax.plot(epochs, history['eval_f1'], 'o-', label='Validation F1', color='orange', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('F1 Score')
        ax.set_title('F1 Score durante el entrenamiento')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Precision y Recall
    if history['eval_precision'] and history['eval_recall']:
        ax = axes[1, 1]
        ax.plot(epochs, history['eval_precision'], 'o-', label='Precision', linewidth=2)
        ax.plot(epochs, history['eval_recall'], 's-', label='Recall', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Score')
        ax.set_title('Precision y Recall durante el entrenamiento')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    history_plot_path = f"{OUTPUT_DIR}batch_size_{batch_size}/training_history.png"
    plt.savefig(history_plot_path, dpi=300, bbox_inches='tight')
    print(f"Grafica guardada en: {history_plot_path}")
    plt.show()

# Visualizar matriz de confusion del mejor modelo
best_exp_name = f'batch_size_{best_batch}'
best_cm = np.array(all_experiments[best_exp_name]['confusion_matrix'])

print(f"\n{'='*80}")
print(f"MATRIZ DE CONFUSION - MEJOR MODELO (Batch Size: {best_batch})")
print(f"{'='*80}")
print(f"\n{'':15} {'Pred Negativo':>15} {'Pred Positivo':>15}")
print(f"{'Real Negativo':<15} {best_cm[0,0]:>15} {best_cm[0,1]:>15}")
print(f"{'Real Positivo':<15} {best_cm[1,0]:>15} {best_cm[1,1]:>15}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'],
            ax=ax)
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusion - Mejor Modelo (Batch Size {best_batch})')
plt.tight_layout()
best_cm_path = f"{OUTPUT_DIR}best_model_confusion_matrix.png"
plt.savefig(best_cm_path, dpi=300, bbox_inches='tight')
print(f"\nMatriz de confusion guardada en: {best_cm_path}")
plt.show()

## 3.10 Verificacion de Archivos para HuggingFace

Verificamos que todos los archivos necesarios esten generados antes de subir.

In [ ]:
# Verificar archivos generados para subir a HuggingFace
import os
from pathlib import Path

print("="*80)
print("VERIFICACION DE ARCHIVOS PARA HUGGINGFACE")
print("="*80)

if 'all_experiments' in locals() and all_experiments:
    best_exp_name = f'batch_size_{best_batch}'
    best_model_dir = all_experiments[best_exp_name]['model_path']
    
    print(f"\nDirectorio del mejor modelo: {best_model_dir}")
    print(f"Batch Size del mejor modelo: {best_batch}")
    print(f"F1 Score: {all_experiments[best_exp_name]['test_results']['f1']:.4f}")
    
    # Verificar archivos necesarios
    required_files = {
        'config.json': 'Configuracion del modelo',
        'pytorch_model.bin': 'Pesos del modelo (PyTorch)',
        'tokenizer_config.json': 'Configuracion del tokenizer',
        'tokenizer.json': 'Tokenizer',
        'special_tokens_map.json': 'Mapa de tokens especiales',
        'sentencepiece.bpe.model': 'Modelo BPE de SentencePiece'
    }
    
    print("\n" + "="*80)
    print("ARCHIVOS REQUERIDOS PARA HUGGINGFACE:")
    print("="*80)
    
    all_present = True
    for filename, description in required_files.items():
        filepath = Path(best_model_dir) / filename
        exists = filepath.exists()
        status = "✓" if exists else "✗"
        size = f"({filepath.stat().st_size / 1024 / 1024:.1f} MB)" if exists else ""
        print(f"{status} {filename:<30} {description:<35} {size}")
        if not exists:
            all_present = False
    
    # Archivos opcionales pero recomendados
    optional_files = {
        'README.md': 'Model card (se genera al subir)',
        'training_args.bin': 'Argumentos de entrenamiento'
    }
    
    print("\n" + "="*80)
    print("ARCHIVOS OPCIONALES:")
    print("="*80)
    
    for filename, description in optional_files.items():
        filepath = Path(best_model_dir) / filename
        exists = filepath.exists()
        status = "✓" if exists else "-"
        size = f"({filepath.stat().st_size / 1024:.1f} KB)" if exists else ""
        print(f"{status} {filename:<30} {description:<35} {size}")
    
    # Resumen
    print("\n" + "="*80)
    print("RESUMEN:")
    print("="*80)
    
    if all_present:
        print("✓ Todos los archivos necesarios están presentes")
        print("✓ El modelo está listo para subir a HuggingFace")
        print("\nPara subirlo:")
        print("1. Descomentar y ejecutar la celda anterior (celda 22)")
        print("2. Asegurarse de estar autenticado en HuggingFace")
        print("3. Verificar que el token tenga permisos de escritura")
    else:
        print("✗ Faltan algunos archivos necesarios")
        print("✗ Asegúrate de que el entrenamiento se completó correctamente")
    
    # Información adicional
    print("\n" + "="*80)
    print("INFORMACIÓN DEL MODELO:")
    print("="*80)
    print(f"Tipo: XLM-RoBERTa Large")
    print(f"Tarea: Clasificación de Sentimientos (Binaria)")
    print(f"Idioma: Español")
    print(f"Dataset: TASS (Twitter)")
    print(f"Clases: 0=Negativo, 1=Positivo")
    print(f"Tamaño del modelo: ~2.2 GB (559M parámetros)")
    
    # Mostrar todos los archivos en el directorio
    print("\n" + "="*80)
    print("ARCHIVOS EN EL DIRECTORIO:")
    print("="*80)
    
    if Path(best_model_dir).exists():
        all_files = sorted(Path(best_model_dir).rglob('*'))
        for filepath in all_files:
            if filepath.is_file():
                relative_path = filepath.relative_to(best_model_dir)
                size_mb = filepath.stat().st_size / 1024 / 1024
                print(f"  {relative_path} ({size_mb:.1f} MB)")
    
else:
    print("\n⚠ El entrenamiento aún no se ha completado")
    print("⚠ Ejecuta primero la celda de entrenamiento (celda 16)")
    print("⚠ Una vez completado, vuelve a ejecutar esta celda para verificar")

print("\n" + "="*80)

In [ ]:
# CONCLUSIONES FINALES

print("="*80)
print("RESUMEN FINAL DE EXPERIMENTOS - TAREA 2 PUNTO 3")
print("="*80)
print()
print(f"Modelo base: {MODEL_NAME}")
print(f"Dataset: TASS (Spanish Twitter Sentiment)")
print(f"Total de muestras: {len(df):,}")
print(f"Distribución: Negativos={(df['label']==0).sum():,} ({(df['label']==0).sum()/len(df)*100:.1f}%), Positivos={(df['label']==1).sum():,} ({(df['label']==1).sum()/len(df)*100:.1f}%)")
print(f"Número de épocas: {NUM_EPOCHS}")
print(f"Batch sizes evaluados: {BATCH_SIZES}")
print(f"Learning rate: {LEARNING_RATE}")
print()

print("="*80)
print("COMPARACIÓN DE RESULTADOS")
print("="*80)
print()
print(comparison_df.to_string(index=False))
print()

print("="*80)
print(f"MEJOR MODELO: Batch Size {best_batch}")
print("="*80)
best_exp = all_experiments[f'batch_size_{best_batch}']
print(f"F1 Score:     {best_exp['test_results']['f1']:.4f}")
print(f"Accuracy:     {best_exp['test_results']['accuracy']:.4f}")
print(f"Precision:    {best_exp['test_results']['precision']:.4f}")
print(f"Recall:       {best_exp['test_results']['recall']:.4f}")
print(f"Specificity:  {best_exp['test_results']['specificity']:.4f}")
print(f"Sensitivity:  {best_exp['test_results']['sensitivity']:.4f}")
print(f"Tiempo total: {best_exp['train_result']['train_runtime']/60:.1f} min")
print()

print("="*80)
print("OBSERVACIONES")
print("="*80)
print()
print("1. EFECTO DEL BATCH SIZE:")
print(f"   - Batch Size más pequeño ({min(BATCH_SIZES)}): Más iteraciones por época, puede capturar mejor los patrones")
print(f"   - Batch Size más grande ({max(BATCH_SIZES)}): Entrenamiento más rápido, gradientes más estables")
print()
print("2. DESBALANCEO DE CLASES:")
print(f"   - Se utilizó weighted loss para manejar el desbalanceo de clases")
print(f"   - Class weights: Negativo={class_weights[0]:.3f}, Positivo={class_weights[1]:.3f}")
print()
print("3. MÉTRICAS BALANCEADAS:")
print(f"   - Se evaluó specificity y sensitivity además de las métricas tradicionales")
print(f"   - El mejor modelo logra un balance entre ambas clases")
print()
print("4. RENDIMIENTO:")
# Calcular mejora porcentual del mejor vs peor
worst_f1 = comparison_df['Test F1'].min()
best_f1 = comparison_df['Test F1'].max()
improvement = ((best_f1 - worst_f1) / worst_f1) * 100
print(f"   - Mejora del mejor modelo respecto al peor: {improvement:.1f}% en F1 Score")
print(f"   - Todos los modelos superan el {comparison_df['Test F1'].min():.3f} de F1 Score")
print()

print("="*80)
print("ARCHIVOS GENERADOS")
print("="*80)
for exp_name, exp_results in all_experiments.items():
    print(f"\n{exp_name.upper()}:")
    print(f"  - Modelo: {exp_results['model_path']}")
    results_file = f"{RESULTS_DIR}punto3_batch_size_{exp_results['batch_size']}_results.json"
    print(f"  - Resultados: {results_file}")
print()
print(f"Comparación CSV: {RESULTS_DIR}punto3_experiments_comparison.csv")
print(f"Gráfica de comparación: {OUTPUT_DIR}experiments_comparison.png")
print()

print("="*80)
print("FIN DEL EXPERIMENTO")
print("="*80)

## 3.11 Subir Modelo a HuggingFace Hub (OPCIONAL)

**IMPORTANTE:** Esta es la ultima celda del notebook. Solo ejecutala si:
- ✓ Has revisado todas las metricas y visualizaciones
- ✓ Estas satisfecho con los resultados del modelo
- ✓ Quieres compartir el modelo publicamente en HuggingFace

Simplemente descomenta el codigo y ejecuta para subir el mejor modelo automaticamente.

In [ ]:
# Subir el mejor modelo a HuggingFace Hub
# Descomentar y ejecutar si deseas subir el modelo

# # Seleccionar el mejor modelo (basado en F1 Score)
# best_exp_name = f'batch_size_{best_batch}'
# best_model_dir = all_experiments[best_exp_name]['model_path']
# best_history = all_experiments[best_exp_name]['history']
# 
# print("="*80)
# print("SUBIENDO MODELO A HUGGINGFACE HUB")
# print("="*80)
# print(f"\nMejor modelo seleccionado: Batch Size {best_batch}")
# print(f"Path del modelo: {best_model_dir}")
# print(f"F1 Score: {all_experiments[best_exp_name]['test_results']['f1']:.4f}")
# print(f"Accuracy: {all_experiments[best_exp_name]['test_results']['accuracy']:.4f}")
# 
# # Cargar el mejor modelo y tokenizer
# print("\nCargando modelo y tokenizer...")
# model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
# tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
# 
# # Configurar el nombre del repositorio en HuggingFace
# repo_name = f"xlm-roberta-large-tass-sentiment-bs{best_batch}"
# print(f"\nNombre del repositorio: {repo_name}")
# 
# # Crear tabla de métricas por época (REQUERIDO POR EL TALLER)
# print("\nGenerando tabla de métricas por época...")
# metrics_table = "\n| Epoch | Loss | Accuracy | F1 Score | Precision | Recall |\n"
# metrics_table += "|-------|------|----------|----------|-----------|--------|\n"
# 
# for i, epoch in enumerate(best_history['epoch']):
#     loss = best_history['eval_loss'][i]
#     acc = best_history['eval_accuracy'][i]
#     f1 = best_history['eval_f1'][i]
#     prec = best_history['eval_precision'][i]
#     rec = best_history['eval_recall'][i]
#     metrics_table += f"| {int(epoch)} | {loss:.4f} | {acc:.4f} | {f1:.4f} | {prec:.4f} | {rec:.4f} |\n"
# 
# # Crear model card (informacion del modelo)
# model_card = f"""
# ---
# language: es
# tags:
# - sentiment-analysis
# - spanish
# - xlm-roberta
# - tass
# - twitter
# datasets:
# - TASS
# metrics:
# - f1
# - accuracy
# - precision
# - recall
# model-index:
# - name: {repo_name}
#   results:
#   - task:
#       type: text-classification
#       name: Sentiment Analysis
#     dataset:
#       name: TASS (Spanish Twitter)
#       type: tass
#     metrics:
#     - type: f1
#       value: {all_experiments[best_exp_name]['test_results']['f1']:.4f}
#       name: F1 Score
#     - type: accuracy
#       value: {all_experiments[best_exp_name]['test_results']['accuracy']:.4f}
#       name: Accuracy
#     - type: precision
#       value: {all_experiments[best_exp_name]['test_results']['precision']:.4f}
#       name: Precision
#     - type: recall
#       value: {all_experiments[best_exp_name]['test_results']['recall']:.4f}
#       name: Recall
# ---
# 
# # XLM-RoBERTa Large Fine-tuned for Spanish Sentiment Analysis (TASS)
# 
# ## Model Description
# 
# This model is a fine-tuned version of [FacebookAI/xlm-roberta-large](https://huggingface.co/FacebookAI/xlm-roberta-large) 
# for sentiment analysis on Spanish Twitter data (TASS dataset).
# 
# ## Training Details
# 
# - **Base Model:** xlm-roberta-large
# - **Task:** Binary Sentiment Classification (Negative/Positive)
# - **Dataset:** TASS (Twitter Analysis Sentiment Seminar)
# - **Training Samples:** {len(train_texts):,}
# - **Validation Samples:** {len(val_texts):,}
# - **Test Samples:** {len(test_texts):,}
# - **Batch Size:** {best_batch}
# - **Epochs:** {NUM_EPOCHS}
# - **Learning Rate:** {LEARNING_RATE}
# - **Weight Decay:** {WEIGHT_DECAY}
# - **Max Sequence Length:** {MAX_LENGTH}
# - **Class Balancing:** Weighted Cross-Entropy Loss
# - **Early Stopping:** Enabled (patience={EARLY_STOPPING_PATIENCE})
# 
# ## Performance (Test Set)
# 
# | Metric | Score |
# |--------|-------|
# | F1 Score | {all_experiments[best_exp_name]['test_results']['f1']:.4f} |
# | Accuracy | {all_experiments[best_exp_name]['test_results']['accuracy']:.4f} |
# | Precision | {all_experiments[best_exp_name]['test_results']['precision']:.4f} |
# | Recall | {all_experiments[best_exp_name]['test_results']['recall']:.4f} |
# | Specificity | {all_experiments[best_exp_name]['test_results']['specificity']:.4f} |
# | Sensitivity | {all_experiments[best_exp_name]['test_results']['sensitivity']:.4f} |
# 
# ## Training History (Validation Set)
# 
# Metrics per epoch during training:
# {metrics_table}
# 
# ## Usage
# 
# ```python
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
# 
# # Cargar modelo y tokenizer
# tokenizer = AutoTokenizer.from_pretrained("tu-usuario/{repo_name}")
# model = AutoModelForSequenceClassification.from_pretrained("tu-usuario/{repo_name}")
# 
# # Ejemplo de uso
# text = "Me encanta este producto, es excelente"
# inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256)
# 
# with torch.no_grad():
#     outputs = model(**inputs)
#     predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
#     predicted_class = torch.argmax(predictions, dim=-1).item()
# 
# labels = {{0: "Negativo", 1: "Positivo"}}
# print(f"Sentimiento: {{labels[predicted_class]}}")
# print(f"Confianza: {{predictions[0][predicted_class].item():.4f}}")
# ```
# 
# ### Como Pipeline
# 
# ```python
# from transformers import pipeline
# 
# # Usar como pipeline
# classifier = pipeline('sentiment-analysis', model='tu-usuario/{repo_name}')
# 
# result = classifier("Me encanta este producto, es excelente")
# print(result)
# # Output: [{{'label': 'LABEL_1', 'score': 0.95}}]
# # LABEL_0 = Negativo, LABEL_1 = Positivo
# ```
# 
# ## Labels
# 
# - `0` (LABEL_0): Negative sentiment
# - `1` (LABEL_1): Positive sentiment
# 
# ## Training Configuration
# 
# The model was trained with weighted loss to handle class imbalance:
# - Negative class weight: {class_weights[0]:.3f}
# - Positive class weight: {class_weights[1]:.3f}
# 
# Distribution in training set:
# - Negative samples: {train_labels.count(0):,} ({train_labels.count(0)/len(train_labels)*100:.1f}%)
# - Positive samples: {train_labels.count(1):,} ({train_labels.count(1)/len(train_labels)*100:.1f}%)
# 
# ## Limitations and Bias
# 
# - This model is specifically trained on Spanish Twitter data
# - Performance may vary on other Spanish text domains
# - The model focuses on binary sentiment (positive/negative) and excludes neutral sentiments
# - May reflect biases present in the TASS Twitter dataset
# 
# ## Citation
# 
# If you use this model, please cite:
# 
# ```
# @misc{{xlm-roberta-tass-sentiment,
#   author = {{Your Name}},
#   title = {{XLM-RoBERTa Large Fine-tuned for Spanish Sentiment Analysis}},
#   year = {{2024}},
#   publisher = {{Hugging Face}},
#   howpublished = {{\\url{{https://huggingface.co/tu-usuario/{repo_name}}}}}
# }}
# ```
# """
# 
# # Guardar model card
# print("Guardando model card...")
# model_card_path = f"{best_model_dir}/README.md"
# with open(model_card_path, 'w', encoding='utf-8') as f:
#     f.write(model_card)
# print(f"✓ Model card guardado en: {model_card_path}")
# 
# # Subir a HuggingFace Hub
# print("\n" + "="*80)
# print("SUBIENDO A HUGGINGFACE HUB...")
# print("="*80)
# try:
#     print("\n[1/2] Subiendo modelo...")
#     model.push_to_hub(repo_name)
#     print("✓ Modelo subido exitosamente")
#     
#     print("\n[2/2] Subiendo tokenizer...")
#     tokenizer.push_to_hub(repo_name)
#     print("✓ Tokenizer subido exitosamente")
#     
#     print("\n" + "="*80)
#     print("✅ SUBIDA COMPLETADA EXITOSAMENTE")
#     print("="*80)
#     print(f"\n🔗 MODELO DISPONIBLE EN:")
#     print(f"   https://huggingface.co/tu-usuario/{repo_name}")
#     print("\n📊 El model card incluye:")
#     print("   ✓ Métricas de desempeño (F1, Accuracy, Precision, Recall)")
#     print("   ✓ Historial completo de entrenamiento por época")
#     print("   ✓ Ejemplos de uso como modelo y como pipeline")
#     print("   ✓ Información de configuración y limitaciones")
#     print("\n💡 Uso como pipeline:")
#     print(f"   from transformers import pipeline")
#     print(f"   classifier = pipeline('sentiment-analysis', model='tu-usuario/{repo_name}')")
#     print(f"   result = classifier('Me encanta este producto')")
#     print("\n" + "="*80)
#     
# except Exception as e:
#     print("\n" + "="*80)
#     print("❌ ERROR AL SUBIR EL MODELO")
#     print("="*80)
#     print(f"\nError: {e}")
#     print("\n🔧 Verifica que:")
#     print("  1. Tu token de HuggingFace tenga permisos de escritura")
#     print("  2. Estés autenticado correctamente (ejecuta celda 3)")
#     print("  3. El nombre del repositorio sea válido")
#     print("  4. No exista ya un repositorio con ese nombre")
#     print("\n💡 Para crear un token con permisos:")
#     print("   https://huggingface.co/settings/tokens")
#     print("="*80)